In [1]:
pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 6.1 MB/s eta 0:00:0000:0100:02
  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'pyspark' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pyspark'. Discussion can be found at https://github.com/pypa/pip/issues/6334
 done
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813798 sha256=5dee79d22e93f9c745da7cede9e070452ddd74d6c0cc28fd6f78d6383558df98
  Stored in directory: /Users/divyanshdoshi/Library/Caches/pip/wheels/00/e3/92/8594f4cee2c9fd4ad82fe85e4bf2559ab8ea84ef19b1dd3d15
Successfully built pyspark
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]m1/2 [pyspark]
Note: you may n

In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import FloatType, IntegerType, StringType
from pyspark.sql import Window
import re, os, json

In [27]:
DATA_DIR = "Documents/GitHub/Cloud_Project/data"
PITSTOPS_XLSX = os.path.join(DATA_DIR, "pitstop.xlsx")
LAP_TIMES_XLSX = os.path.join(DATA_DIR, "lap_times.xlsx")
SAFETYCAR_XLSX = os.path.join(DATA_DIR, "safety_cars.xlsx")
REDFLAG_XLSX = os.path.join(DATA_DIR, "red_flags.xlsx")
OUTPUT_PARQUET = "output/f1_full_engineered.parquet"
OUTPUT_CSV = "output/f1_full_engineered_csv"
ANONYMIZE_DRIVERS = True

# Print the actual paths to verify
print("File paths:")
print(f"Pitstops: {PITSTOPS_XLSX}")
print(f"Lap Times: {LAP_TIMES_XLSX}")
print(f"Safety Car: {SAFETYCAR_XLSX}")
print(f"Red Flag: {REDFLAG_XLSX}")

File paths:
Pitstops: Documents/GitHub/Cloud_Project/data/pitstop.xlsx
Lap Times: Documents/GitHub/Cloud_Project/data/lap_times.xlsx
Safety Car: Documents/GitHub/Cloud_Project/data/safety_cars.xlsx
Red Flag: Documents/GitHub/Cloud_Project/data/red_flags.xlsx


In [29]:
conda install -c conda-forge openjdk=17

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
doneecting package metadata (repodata.json): - 
Solving envdone


==> WARNING: A newer version of conda exists. <==
    current version: 25.7.0
    latest version: 25.9.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /opt/anaconda3

  added / updated specs:
    - openjdk=17


The following packages will be UPDATED:

  openjdk            conda-forge::openjdk-11.0.9.1-h069f3c~ --> pkgs/main::openjdk-17.0.14-h80987f9_0 




donearing transaction: | 
donefying transaction: - 
Executing transactiondone

Note: you may need to restart the kernel to use updated packages.


In [30]:
spark = (SparkSession.builder
         .appName("F1FullPipeline")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

In [31]:
def normalize_text(s):
    if s is None: return None
    s = re.sub(r"[^\x00-\x7F]+","",str(s))
    s = re.sub(r"\s+"," ", s)
    return s.strip()
normalize_udf = udf(normalize_text, StringType())

def parse_laptime_to_seconds(t):
    if t is None: return None
    t = str(t).strip()
    if t=="": return None
    try: return float(t)
    except: pass
    m = re.match(r"(?:(\d+):)?(\d+)(?:[:\.](\d+))?$", t)
    if not m: return None
    g = m.groups()
    minutes = int(g[0]) if g[0] else 0
    seconds = int(g[1])
    ms = float("0."+g[2]) if g[2] else 0
    return minutes*60 + seconds + ms
parse_laptime_udf = udf(parse_laptime_to_seconds, FloatType())

In [32]:
def read_excel_if_exists(path):
    return spark.read.option("header", True).option("inferSchema", True).format("excel").load(path) if os.path.exists(path) else None

pit = read_excel_if_exists(PITSTOPS_XLSX)
laps = read_excel_if_exists(LAP_TIMES_XLSX)
safety = read_excel_if_exists(SAFETYCAR_XLSX)
redflag = read_excel_if_exists(REDFLAG_XLSX)

In [33]:
from pyspark.sql.functions import col, struct, collect_list, avg, sum, first, when, broadcast

if pit is not None:
    pit_clean = (pit.withColumn("race_norm", normalize_udf(col("Race Name")))
                 .withColumn("driver_norm", normalize_udf(col("Driver")))
                 .withColumn("pit_time_s", col("Pit_Time").cast(FloatType()))
                 .withColumn("stint_num", col("Stint").cast(IntegerType()))
                 .withColumn("Stint_Length_num", col("Stint Length").cast(IntegerType()))
                )
    stint_struct = struct(col("stint_num").alias("stint"),
                          col("Tire Compound").alias("tire"),
                          col("Stint_Length_num").alias("length"),
                          col("pit_time_s").alias("pit_time"))
    group_cols = ["Season","Round","race_norm","driver_norm","Constructor"]
    pit_merged = pit_clean.groupBy(*group_cols).agg(
        collect_list(stint_struct).alias("stints"),
        avg("pit_time_s").alias("avg_pit_time"),
        sum("pit_time_s").alias("total_pit_time"),
        first("Circuit").alias("Circuit"),
        first("Country").alias("Country")
    )

# Fixed indentation for safety and redflag
if safety is not None:
    safety = safety.withColumn("race_norm", normalize_udf(col("Race")))
if redflag is not None:
    redflag = redflag.withColumn("race_norm", normalize_udf(col("Race")))

if pit_merged is not None:
    from pyspark.sql.functions import broadcast
    joined = pit_merged
    if safety is not None:
        joined = joined.join(broadcast(safety), "race_norm", "left")
    if redflag is not None:
        joined = joined.join(broadcast(redflag), "race_norm", "left")
    joined = joined.withColumn("had_safety_car", when(col("FullLaps").isNotNull(), True).otherwise(False)) \
                   .withColumn("had_red_flag", when(col("Lap").isNotNull(), True).otherwise(False))

❌ No pit data available for processing
